<a href="https://colab.research.google.com/github/chitramadarakhandi/genai_assignment4_unit4/blob/main/pes2ug23cs157_unit4_assignment_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unit 4 Assignment: Self-Evaluating Agentic RAG System

**System Overview:**
- **Agent 1 (RAG Retriever):** Retrieves relevant context from FAISS vector store and generates an answer
- **Agent 2 (Quality Evaluator):** Scores the answer using DeepEval (Faithfulness + AnswerRelevancy)
- **Agent 3 (Revisor):** If FAIL, revises the answer using the evaluator's feedback

**Topic:** Climate Change & Global Warming (scientific facts from multiple sub-topics)

## Step 0: Install Dependencies

In [43]:
# Install all required packages
# Colab may show dependency-conflict warnings (google-adk, opentelemetry, pydantic) — safe to ignore.
!pip install -q crewai crewai-tools langchain langchain-community langchain-groq \
    faiss-cpu sentence-transformers deepeval openai tiktoken \
    langchain-huggingface 2>&1 | grep -E '(ERROR|Successfully installed)' || true

print("\n✅ Packages installed. Dependency conflict warnings from Colab's pre-installed packages are safe to ignore.")


✅ Packages installed. Dependency conflict warnings from Colab's pre-installed packages are safe to ignore.


## Step 1: Enter API Keys

Run this cell and **type your keys into the secure input boxes** that appear below.

| Key | Where to get it |
|-----|-----------------|
| `GROQ_API_KEY` | https://console.groq.com → API Keys |
| `OPENAI_API_KEY` | https://platform.openai.com → API Keys (DeepEval uses `gpt-4o-mini` internally) |

> 🔒 `getpass` hides your input — nothing is printed or stored in the notebook.

In [44]:
import os
from getpass import getpass

print("Enter your API keys below (input is hidden):")
print("-" * 45)

groq_key   = getpass("🔑 GROQ_API_KEY   : ")
openai_key = getpass("🔑 OPENAI_API_KEY : ")

if not groq_key.strip():
    raise ValueError("GROQ_API_KEY cannot be empty. Get one at https://console.groq.com")
if not openai_key.strip():
    raise ValueError("OPENAI_API_KEY cannot be empty. Get one at https://platform.openai.com")

os.environ["GROQ_API_KEY"]   = groq_key.strip()
os.environ["OPENAI_API_KEY"] = openai_key.strip()

print("-" * 45)
print(f"✅ GROQ_API_KEY   set ({len(groq_key)} chars)")
print(f"✅ OPENAI_API_KEY set ({len(openai_key)} chars)")
print("Keys loaded into environment — ready to proceed!")

Enter your API keys below (input is hidden):
---------------------------------------------
🔑 GROQ_API_KEY   : ··········
🔑 OPENAI_API_KEY : ··········
---------------------------------------------
✅ GROQ_API_KEY   set (56 chars)
✅ OPENAI_API_KEY set (164 chars)
Keys loaded into environment — ready to proceed!


---
## Part 1: Knowledge Base

**Topic: Climate Change & Global Warming**

I chose this topic because it is rich in verifiable scientific facts, covers multiple sub-domains (atmospheric science, oceanography, policy, biology), and is publicly well-documented — making it ideal for testing RAG faithfulness and answer relevancy. The knowledge base contains 600+ words covering 8+ distinct facts about causes, effects, measurements, and mitigation strategies.

In [45]:
pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface faiss-cpu

In [46]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

In [47]:
# langchain.text_splitter moved to langchain_text_splitters in LangChain >= 0.2
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# ── Knowledge Base Text ─────────────────────────────────────────────────────
KNOWLEDGE_BASE = """
Climate Change and Global Warming: A Comprehensive Overview

Climate change refers to long-term shifts in global temperatures and weather patterns.
While some climate change is natural, since the mid-20th century, human activities have
been the main driver of climate change, primarily due to the burning of fossil fuels
like coal, oil, and natural gas.

Greenhouse Gases and the Greenhouse Effect:
The greenhouse effect is the process by which gases in Earth's atmosphere trap heat from
the sun. Carbon dioxide (CO2), methane (CH4), nitrous oxide (N2O), and water vapor are
the primary greenhouse gases. CO2 levels have risen from approximately 280 parts per
million (ppm) in pre-industrial times to over 420 ppm as of 2023, the highest in over
3 million years. Methane is about 80 times more potent than CO2 over a 20-year period
but remains in the atmosphere for a shorter duration.

Global Temperature Rise:
Earth's average surface temperature has risen by approximately 1.1 degrees Celsius
(1.9°F) above pre-industrial levels as of 2023. The Intergovernmental Panel on Climate
Change (IPCC) warns that limiting warming to 1.5°C requires global emissions to reach
net zero by approximately 2050. The last decade (2011-2020) was the warmest on record.
The 10 warmest years on record have all occurred since 2005.

Melting Ice and Rising Sea Levels:
Arctic sea ice has declined by approximately 13% per decade since 1979. The Greenland
ice sheet is losing about 280 billion tonnes of ice per year. Global sea levels have
risen about 20 cm (8 inches) since 1900, with the rate accelerating to over 3.6 mm per
year in recent decades. If all ice on Earth were to melt, sea levels would rise by an
estimated 65-70 meters, flooding most coastal cities.

Ocean Acidification:
The oceans absorb about 30% of the CO2 produced by human activities. As CO2 dissolves
in seawater, it forms carbonic acid, lowering the ocean's pH. Ocean pH has dropped from
8.2 to 8.1 since pre-industrial times — a 26% increase in acidity. This acidification
threatens coral reefs, shellfish, and marine ecosystems broadly. Coral bleaching events
have doubled in frequency since the 1980s due to warming ocean temperatures.

Extreme Weather Events:
Climate change intensifies extreme weather events. Warmer oceans fuel more powerful
hurricanes and typhoons. Higher temperatures increase the risk of wildfires by drying
out vegetation. More intense rainfall events cause flooding, while prolonged droughts
reduce agricultural productivity. Studies show that climate change made the 2021
Pacific Northwest heat dome at least 150 times more likely.

Impacts on Biodiversity:
Up to 1 million plant and animal species are at risk of extinction due to climate change
and related habitat destruction, according to a 2019 UN report. Coral reefs, which
support 25% of all marine life, could decline by 70-90% with 1.5°C of warming and
virtually disappear at 2°C. Many species are shifting their ranges poleward or to higher
altitudes in response to rising temperatures.

Mitigation Strategies:
Renewable energy sources — solar, wind, hydro, and geothermal — are the primary tools
for reducing CO2 emissions. Solar energy capacity has grown by over 400% in the past
decade. Electric vehicles (EVs) are replacing internal combustion engines; EV sales
reached over 10 million units globally in 2022. Carbon capture and storage (CCS)
technology can trap CO2 emissions from power plants and industrial sources. Reforestation
and afforestation can act as natural carbon sinks — forests currently absorb about
2.6 billion tonnes of CO2 per year.

International Climate Agreements:
The Paris Agreement, adopted in 2015 and signed by 196 parties, aims to limit global
warming to well below 2°C above pre-industrial levels, with efforts to limit it to 1.5°C.
Countries submit Nationally Determined Contributions (NDCs) outlining their climate
action plans. The Kyoto Protocol (1997) was the first binding international treaty
requiring developed nations to reduce emissions. COP28 held in Dubai in 2023 was a key
milestone where nations agreed to transition away from fossil fuels.

Economic Impacts:
The economic cost of climate change could reach 10-23% of global GDP by 2100 if warming
reaches 3°C, according to various economic models. The Stern Review estimated the cost
of inaction at 5-20% of global GDP annually, while the cost of action is only about 1%
of GDP per year. Climate-related disasters already cost the global economy over
$300 billion annually in recent years.
"""

# ── Text Splitting ───────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    separators=["\n\n", "\n", ".", " "]
)
chunks = splitter.create_documents([KNOWLEDGE_BASE])
print(f"Total chunks created: {len(chunks)}")
for i, c in enumerate(chunks):
    print(f"  Chunk {i+1} ({len(c.page_content)} chars): {c.page_content[:80]}...")

Total chunks created: 19
  Chunk 1 (353 chars): Climate Change and Global Warming: A Comprehensive Overview

Climate change refe...
  Chunk 2 (390 chars): Greenhouse Gases and the Greenhouse Effect:
The greenhouse effect is the process...
  Chunk 3 (139 chars): 3 million years. Methane is about 80 times more potent than CO2 over a 20-year p...
  Chunk 4 (367 chars): Global Temperature Rise:
Earth's average surface temperature has risen by approx...
  Chunk 5 (60 chars): The 10 warmest years on record have all occurred since 2005....
  Chunk 6 (378 chars): Melting Ice and Rising Sea Levels:
Arctic sea ice has declined by approximately ...
  Chunk 7 (53 chars): estimated 65-70 meters, flooding most coastal cities....
  Chunk 8 (368 chars): Ocean Acidification:
The oceans absorb about 30% of the CO2 produced by human ac...
  Chunk 9 (76 chars): have doubled in frequency since the 1980s due to warming ocean temperatures....
  Chunk 10 (360 chars): Extreme Weather Events:
Climate change int

In [48]:
# ── Build FAISS Vector Store ─────────────────────────────────────────────────
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(chunks, embeddings)
print("FAISS vector store built successfully.")
print(f"Vector store contains {vector_store.index.ntotal} vectors.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS vector store built successfully.
Vector store contains 19 vectors.


---

---

## Part 2: RAG Agent

In [49]:
!pip install -q langchain-groq

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    max_tokens=1024,
    timeout=60  # Increased timeout to 60 seconds
)
print("Groq LLM initialized.")

Groq LLM initialized.


In [50]:
# Step 1: Install everything with correct pinned versions
!pip install -q "crewai[litellm]" "litellm>=1.0.0" "pydantic==2.11.9" "python-dotenv==1.1.1" "tiktoken==0.8.0"

In [51]:
!pip install -q litellm

import os
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from langchain_groq import ChatGroq

# ── LLM for direct use inside tools (LangChain object) ──────────────────────
llm_chain = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    max_tokens=1024,
    api_key=os.environ["GROQ_API_KEY"],
    timeout=60  # Increased timeout to 60 seconds
)

# ── LLM string for CrewAI agents (needs litellm installed) ──────────────────
LLM_MODEL = "groq/llama-3.3-70b-versatile"

# ── Tool: FAISS Retriever ────────────────────────────────────────────────────
@tool("faiss_retriever")
def faiss_retriever(question: str) -> str:
    """Retrieves the most relevant context chunks from the FAISS vector store
    for a given question. Returns both the retrieved context and the answer
    in a structured format."""
    docs = vector_store.similarity_search(question, k=4)
    context = "\n\n".join([d.page_content for d in docs])

    prompt = f"""You are a helpful assistant. Answer the question ONLY using the provided context.
If the answer is not in the context, say "I don't have information about this in my knowledge base."

Context:
{context}

Question: {question}

Answer:"""

    response = llm_chain.invoke(prompt)
    answer = response.content.strip()

    return f"""ANSWER: {answer}

RETRIEVED_CONTEXT: {context}"""


# ── Agent 1: RAG Retriever ───────────────────────────────────────────────────
rag_agent = Agent(
    role="RAG Retriever",
    goal="Retrieve relevant context from the vector store and generate a faithful, accurate answer to the user's question.",
    backstory=(
        "You are an expert information retrieval specialist. Your job is to use the FAISS "
        "vector store tool to find the most relevant passages and generate a precise, "
        "well-grounded answer. Always include both the answer and the retrieved context in your output."
    ),
    tools=[faiss_retriever],
    llm=LLM_MODEL,
    verbose=True
)

print("RAG Agent created.")

RAG Agent created.


In [52]:
# ── Quick test of RAG tool before building full pipeline ────────────────────
print("=" * 60)
print("Quick RAG tool test:")
print("=" * 60)
test_result = faiss_retriever.run("What is the current level of CO2 in the atmosphere?")
print(test_result[:600])
print("...")

Quick RAG tool test:
ANSWER: Over 420 parts per million (ppm) as of 2023.

RETRIEVED_CONTEXT: Greenhouse Gases and the Greenhouse Effect:
The greenhouse effect is the process by which gases in Earth's atmosphere trap heat from
the sun. Carbon dioxide (CO2), methane (CH4), nitrous oxide (N2O), and water vapor are
the primary greenhouse gases. CO2 levels have risen from approximately 280 parts per
million (ppm) in pre-industrial times to over 420 ppm as of 2023, the highest in over

3 million years. Methane is about 80 times more potent than CO2 over a 20-year period
but remains in the atmosphere for a shorter durat
...


---
## Part 3: Quality Evaluator Agent

In [53]:
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
import json

EVAL_THRESHOLD = 0.7

# ── Tool: DeepEval Evaluator ─────────────────────────────────────────────────
@tool("deepeval_evaluator")
def deepeval_evaluator(rag_output: str) -> str:
    """Evaluates an RAG answer using DeepEval FaithfulnessMetric and
    AnswerRelevancyMetric. Input must be the full RAG output string containing
    ANSWER: and RETRIEVED_CONTEXT: sections. Returns scores, pass/fail verdict,
    and detailed reasons."""
    try:
        if "ANSWER:" in rag_output and "RETRIEVED_CONTEXT:" in rag_output:
            parts = rag_output.split("RETRIEVED_CONTEXT:")
            answer_part = parts[0].replace("ANSWER:", "").strip()
            context_part = parts[1].strip()
        else:
            answer_part = rag_output.strip()
            context_part = rag_output.strip()

        question = "What information is being requested?"

        test_case = LLMTestCase(
            input=question,
            actual_output=answer_part,
            retrieval_context=[context_part]
        )

        faithfulness_metric = FaithfulnessMetric(
            threshold=EVAL_THRESHOLD,
            model="gpt-4o-mini",
            include_reason=True
        )
        relevancy_metric = AnswerRelevancyMetric(
            threshold=EVAL_THRESHOLD,
            model="gpt-4o-mini",
            include_reason=True
        )

        faithfulness_metric.measure(test_case)
        relevancy_metric.measure(test_case)

        faith_score  = faithfulness_metric.score
        rel_score    = relevancy_metric.score
        faith_reason = faithfulness_metric.reason or "No reason provided"
        rel_reason   = relevancy_metric.reason or "No reason provided"

        verdict = "PASS" if (faith_score >= EVAL_THRESHOLD and rel_score >= EVAL_THRESHOLD) else "FAIL"
        reasons = []
        if faith_score < EVAL_THRESHOLD:
            reasons.append(f"Faithfulness below threshold: {faith_reason}")
        if rel_score < EVAL_THRESHOLD:
            reasons.append(f"Answer Relevancy below threshold: {rel_reason}")

        result = {
            "faithfulness": round(faith_score, 3),
            "relevancy": round(rel_score, 3),
            "verdict": verdict,
            "reasons": reasons if reasons else ["Both metrics above threshold."],
            "answer": answer_part,
            "context": context_part
        }
        return json.dumps(result, indent=2)

    except Exception as e:
        return json.dumps({
            "faithfulness": 0.0,
            "relevancy": 0.0,
            "verdict": "FAIL",
            "reasons": [f"Evaluation error: {str(e)}"],
            "answer": rag_output[:300],
            "context": ""
        }, indent=2)


# ── Agent 2: Quality Evaluator ───────────────────────────────────────────────
evaluator_agent = Agent(
    role="Quality Evaluator",
    goal="Evaluate the faithfulness and relevancy of the RAG answer using DeepEval metrics and produce a structured verdict.",
    backstory=(
        "You are a rigorous QA specialist who evaluates AI-generated answers. "
        "You use DeepEval's FaithfulnessMetric and AnswerRelevancyMetric to score answers. "
        "You always provide clear, structured feedback including scores, pass/fail verdict, "
        "and specific reasons for any failures."
    ),
    tools=[deepeval_evaluator],
    llm=LLM_MODEL,   # <-- FIXED: string not ChatGroq object
    verbose=True
)

print("✅ Evaluator Agent created.")

✅ Evaluator Agent created.


---
## Part 4: Revisor Agent

In [54]:
# ── Tool: Answer Revisor ─────────────────────────────────────────────────────
@tool("answer_revisor")
def answer_revisor(revision_input: str) -> str:
    """Revises a failed RAG answer based on evaluator feedback.
    Input must be a JSON string with keys: question, original_answer, context, failure_reasons.
    Returns a corrected, context-grounded answer."""
    try:
        data = json.loads(revision_input)
        question        = data.get("question", "")
        original_answer = data.get("original_answer", "")
        context         = data.get("context", "")
        failure_reasons = data.get("failure_reasons", [])
    except json.JSONDecodeError:
        question        = "Revise the answer"
        original_answer = revision_input
        context         = revision_input
        failure_reasons = ["Evaluation failed"]

    reasons_text = "\n".join(f"- {r}" for r in failure_reasons)

    prompt = f"""You are a careful, accurate assistant. A previous answer was flagged as poor quality.

Original Question: {question}

Original (Failed) Answer:
{original_answer}

Failure Reasons from Evaluator:
{reasons_text}

Retrieved Context (use ONLY this to revise):
{context}

Instructions:
1. Address EVERY failure reason listed above.
2. Base your revised answer ONLY on the provided context — do not add external information.
3. Be concise, accurate, and directly responsive to the question.
4. If the context does not contain the answer, clearly state that.

Revised Answer:"""

    response = llm_chain.invoke(prompt)   # <-- FIXED: llm_chain not llm
    return response.content.strip()


# ── Agent 3: Revisor ─────────────────────────────────────────────────────────
revisor_agent = Agent(
    role="Answer Revisor",
    goal="Revise failed answers to improve faithfulness and relevancy, strictly grounded in the retrieved context.",
    backstory=(
        "You are an expert editor who specializes in correcting AI-generated answers. "
        "When given a failed answer and the evaluator's feedback, you produce a revised answer "
        "that directly addresses each failure reason, using only the provided context."
    ),
    tools=[answer_revisor],
    llm=LLM_MODEL,        # <-- FIXED: string not ChatGroq object
    verbose=True
)

print("✅ Revisor Agent created.")

✅ Revisor Agent created.


---
## Part 5: Full Pipeline

The pipeline function below:
1. Runs RAG Agent to get answer + context
2. Runs Evaluator Agent to score
3. If FAIL → runs Revisor Agent and re-evaluates
4. Returns the final scores and answer

In [55]:
import json
import re

def parse_eval_result(eval_output: str) -> dict:
    """Extract the JSON evaluation result from the evaluator's output."""
    # Try to find a JSON block in the output
    json_match = re.search(r'\{[\s\S]*\}', eval_output)
    if json_match:
        try:
            return json.loads(json_match.group())
        except:
            pass
    # Defaults if parsing fails
    return {
        "faithfulness": 0.0,
        "relevancy": 0.0,
        "verdict": "FAIL",
        "reasons": ["Could not parse evaluation output"],
        "answer": "",
        "context": ""
    }


def run_pipeline(question: str) -> dict:
    """Run the full 3-agent RAG pipeline for a single question."""
    print(f"\n{'='*70}")
    print(f"QUESTION: {question}")
    print('='*70)

    # ── Task 1: RAG Retrieval ────────────────────────────────────────────────
    rag_task = Task(
        description=f"""
        Use the faiss_retriever tool to answer the following question:
        "{question}"

        Your final output MUST be formatted exactly as:
        ANSWER: <your answer here>

        RETRIEVED_CONTEXT: <the context chunks used>
        """,
        expected_output="A string containing ANSWER: and RETRIEVED_CONTEXT: sections.",
        agent=rag_agent
    )

    # ── Task 2: Evaluation ───────────────────────────────────────────────────
    eval_task = Task(
        description="""
        Take the output from the RAG Retriever task (which contains ANSWER: and RETRIEVED_CONTEXT: sections)
        and pass it to the deepeval_evaluator tool.
        Report the exact JSON output from the tool including faithfulness score, relevancy score,
        verdict (PASS/FAIL), and reasons.
        """,
        expected_output="A JSON object with faithfulness, relevancy, verdict, reasons, answer, and context.",
        agent=evaluator_agent,
        context=[rag_task]
    )

    # ── Run RAG + Eval Crew ──────────────────────────────────────────────────
    crew1 = Crew(
        agents=[rag_agent, evaluator_agent],
        tasks=[rag_task, eval_task],
        process=Process.sequential,
        verbose=False
    )
    crew1_result = crew1.kickoff()

    # Parse evaluation result
    eval_output_str = str(crew1_result.raw) if hasattr(crew1_result, 'raw') else str(crew1_result)
    eval_data = parse_eval_result(eval_output_str)

    initial_faithfulness = eval_data.get("faithfulness", 0.0)
    initial_relevancy    = eval_data.get("relevancy", 0.0)
    initial_verdict      = eval_data.get("verdict", "FAIL")
    failure_reasons      = eval_data.get("reasons", [])
    original_answer      = eval_data.get("answer", "")
    context_used         = eval_data.get("context", "")

    print(f"\n[INITIAL EVALUATION]")
    print(f"  Faithfulness : {initial_faithfulness}")
    print(f"  Relevancy    : {initial_relevancy}")
    print(f"  Verdict      : {initial_verdict}")

    final_faithfulness = initial_faithfulness
    final_relevancy    = initial_relevancy
    revised_answer     = None

    # ── Revision if FAIL ────────────────────────────────────────────────────
    if initial_verdict == "FAIL":
        print("\n[FAIL DETECTED — Running Revisor Agent]")

        revision_input = json.dumps({
            "question": question,
            "original_answer": original_answer,
            "context": context_used,
            "failure_reasons": failure_reasons
        })

        revision_task = Task(
            description=f"""
            The RAG answer for the question "{question}" failed quality evaluation.
            Use the answer_revisor tool with the following JSON input:
            {revision_input}

            Produce a revised answer that addresses the failure reasons.
            Your final output must be the revised answer text only.
            """,
            expected_output="The revised, improved answer text.",
            agent=revisor_agent
        )

        crew2 = Crew(
            agents=[revisor_agent],
            tasks=[revision_task],
            process=Process.sequential,
            verbose=False
        )
        crew2_result = crew2.kickoff()
        revised_answer = str(crew2_result.raw) if hasattr(crew2_result, 'raw') else str(crew2_result)

        print(f"\n[REVISED ANSWER]\n{revised_answer[:400]}...")

        # Re-evaluate the revised answer
        revised_rag_output = f"ANSWER: {revised_answer}\n\nRETRIEVED_CONTEXT: {context_used}"
        re_eval_raw = deepeval_evaluator(revised_rag_output)
        try:
            re_eval_data = json.loads(re_eval_raw)
        except:
            re_eval_data = {"faithfulness": 0.0, "relevancy": 0.0}

        final_faithfulness = re_eval_data.get("faithfulness", 0.0)
        final_relevancy    = re_eval_data.get("relevancy", 0.0)

        print(f"\n[POST-REVISION EVALUATION]")
        print(f"  Faithfulness : {final_faithfulness}")
        print(f"  Relevancy    : {final_relevancy}")

    return {
        "question": question,
        "initial_faithfulness": initial_faithfulness,
        "initial_relevancy": initial_relevancy,
        "initial_verdict": initial_verdict,
        "final_faithfulness": final_faithfulness,
        "final_relevancy": final_relevancy,
        "original_answer": original_answer,
        "revised_answer": revised_answer
    }

print("Pipeline function defined.")

Pipeline function defined.


In [56]:
# ── Test Questions (from Knowledge Base) ────────────────────────────────────
test_questions = [
    "What is the current level of CO2 in the atmosphere compared to pre-industrial times?",
    "How much has global sea level risen since 1900?",
    "What are the main mitigation strategies for climate change?",
    "How does ocean acidification affect marine life?",
    "What did the Paris Agreement aim to achieve?"
]

# ── Adversarial Questions (answers NOT in knowledge base) ───────────────────
adversarial_questions = [
    "What is the latest iPhone model and its price?",
    "Who won the FIFA World Cup in 2022?"
]

all_questions = test_questions + adversarial_questions
print(f"Total questions to run: {len(all_questions)}")

Total questions to run: 7


In [57]:
import traceback
import sys # Ensure sys is imported for file=sys.stderr, although we're moving away from it for printing

# ── Execute Full Pipeline ────────────────────────────────────────────────────
# NOTE: This cell may take 5-15 minutes due to DeepEval API calls.

results = []

for q in all_questions:
    try:
        result = run_pipeline(q)
        results.append(result)
    except Exception as e:
        error_traceback = traceback.format_exc()
        print(f"\nERROR on question '{q}': {e}") # Print to stdout
        print("\n--- FULL TRACEBACK ---")
        print(error_traceback)
        print("----------------------\n")

        results.append({
            "question": q,
            "initial_faithfulness": 0.0,
            "initial_relevancy": 0.0,
            "initial_verdict": "ERROR",
            "final_faithfulness": 0.0,
            "final_relevancy": 0.0,
            "original_answer": "",
            "revised_answer": None,
            "error_traceback": error_traceback # Store traceback in results
        })

print("\n" + "="*70)
print("ALL QUESTIONS PROCESSED")
print("="*70)


QUESTION: What is the current level of CO2 in the atmosphere compared to pre-industrial times?


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retriever                                                                                           │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Use the faiss_retriever tool to answer the following question:                                         │
│          "What is the current level of CO2 in the atmosphere compared to pre-industrial times?"                 │
│                                                                                                                 │
│          Your final output MUST be formatted exactly as:                                                        │
│          ANSWER: <your answer here>                                                                             │
│                                                                                                                 │
│          RETRIEVED_CONTEXT: <the context chunks used>                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool faiss_retriever executed with result: ANSWER: CO2 levels have risen from approximately 280 parts per million (ppm) in pre-industrial times to over 420 ppm as of 2023.

RETRIEVED_CONTEXT: 3 million years. Methane is about 80 times more pot...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retriever                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ANSWER: CO2 levels have risen from approximately 280 parts per million (ppm) in pre-industrial times to over   │
│  420 ppm as of 2023.                                                                                            │
│                                                                                                                 │
│  RETRIEVED_CONTEXT: 3 million years. Methane is about 80 times more potent than CO2 over a 20-year period       │
│  but remains in the atmosphere for a shorter duration.                                                          │
│                                                                                                                 │
│  technology can trap CO2 emissions from power plants and industrial sources. Reforestation                      │
│  and afforestation can act as natural carbon sinks — forests currently absorb about                             │
│  2.6 billion tonnes of CO2 per year.                                                                            │
│                                                                                                                 │
│  Greenhouse Gases and the Greenhouse Effect:                                                                    │
│  The greenhouse effect is the process by which gases in Earth's atmosphere trap heat from                       │
│  the sun. Carbon dioxide (CO2), methane (CH4), nitrous oxide (N2O), and water vapor are                         │
│  the primary greenhouse gases. CO2 levels have risen from approximately 280 parts per                           │
│  million (ppm) in pre-industrial times to over 420 ppm as of 2023, the highest in over                          │
│                                                                                                                 │
│  Global Temperature Rise:                                                                                       │
│  Earth's average surface temperature has risen by approximately 1.1 degrees Celsius                             │
│  (1.9°F) above pre-industrial levels as of 2023. The Intergovernmental Panel on Climate                         │
│  Change (IPCC) warns that limiting warming to 1.5°C requires global emissions to reach                          │
│  net zero by approximately 2050. The last decade (2011-2020) was the warmest on record.                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_completed' closed 'llm_call_started' (expected 
'agent_execution_started')

...


ERROR on question 'What did the Paris Agreement aim to achieve?': litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01khxtcp90fht9zgt3tyc083hf` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11693, Requested 2289. Please try again in 9.91s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


--- FULL TRACEBACK ---
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/litellm/llms/custom_httpx/llm_http_handler.py", line 235, in _make_common_sync_call
    response = sync_httpx_client.post(
               ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/litellm/llms/custom_httpx/http_handler.py", line 1047, in post
    raise e
  File "/usr/local/lib/python3.12/dist-packages/litellm/llms/custom_httpx/http_handler.py", line 1029

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retriever                                                                                           │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Use the faiss_retriever tool to answer the following question:                                         │
│          "What is the latest iPhone model and its price?"                                                       │
│                                                                                                                 │
│          Your final output MUST be formatted exactly as:                                                        │
│          ANSWER: <your answer here>                                                                             │
│                                                                                                                 │
│          RETRIEVED_CONTEXT: <the context chunks used>                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


ERROR on question 'What is the latest iPhone model and its price?': litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01khxtcp90fht9zgt3tyc083hf` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11667, Requested 2441. Please try again in 10.54s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


--- FULL TRACEBACK ---
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/litellm/llms/custom_httpx/llm_http_handler.py", line 235, in _make_common_sync_call
    response = sync_httpx_client.post(
               ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/litellm/llms/custom_httpx/http_handler.py", line 1047, in post
    raise e
  File "/usr/local/lib/python3.12/dist-packages/litellm/llms/custom_httpx/http_handler.py", line 1

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retriever                                                                                           │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Use the faiss_retriever tool to answer the following question:                                         │
│          "Who won the FIFA World Cup in 2022?"                                                                  │
│                                                                                                                 │
│          Your final output MUST be formatted exactly as:                                                        │
│          ANSWER: <your answer here>                                                                             │
│                                                                                                                 │
│          RETRIEVED_CONTEXT: <the context chunks used>                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


ERROR on question 'Who won the FIFA World Cup in 2022?': litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01khxtcp90fht9zgt3tyc083hf` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11619, Requested 2672. Please try again in 11.455s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


--- FULL TRACEBACK ---
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/litellm/llms/custom_httpx/llm_http_handler.py", line 235, in _make_common_sync_call
    response = sync_httpx_client.post(
               ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/litellm/llms/custom_httpx/http_handler.py", line 1047, in post
    raise e
  File "/usr/local/lib/python3.12/dist-packages/litellm/llms/custom_httpx/http_handler.py", line 1029, in po

In [58]:
import pandas as pd

# ── Results Table ────────────────────────────────────────────────────────────
rows = []
for r in results:
    rows.append({
        "Question (truncated)": r["question"][:60] + "...",
        "Initial Faith.": r["initial_faithfulness"],
        "Initial Rel.": r["initial_relevancy"],
        "Verdict": r["initial_verdict"],
        "Final Faith.": r["final_faithfulness"],
        "Final Rel.": r["final_relevancy"],
        "Revised?": "Yes" if r["revised_answer"] else "No"
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# ── Pass Rate Summary ────────────────────────────────────────────────────────
initial_passes = sum(1 for r in results if r["initial_verdict"] == "PASS")
final_passes   = sum(
    1 for r in results
    if r["final_faithfulness"] >= EVAL_THRESHOLD and r["final_relevancy"] >= EVAL_THRESHOLD
)
total = len(results)

print(f"\n{'='*50}")
print(f"Initial Pass Rate: {initial_passes}/{total} ({100*initial_passes/total:.1f}%)")
print(f"Final Pass Rate  : {final_passes}/{total} ({100*final_passes/total:.1f}%)")
print(f"{'='*50}")

                                           Question (truncated)  Initial Faith.  Initial Rel. Verdict  Final Faith.  Final Rel. Revised?
What is the current level of CO2 in the atmosphere compared ...             0.0           0.0   ERROR           0.0         0.0       No
             How much has global sea level risen since 1900?...             0.0           0.0   ERROR           0.0         0.0       No
 What are the main mitigation strategies for climate change?...             0.0           0.0   ERROR           0.0         0.0       No
            How does ocean acidification affect marine life?...             0.0           0.0   ERROR           0.0         0.0       No
                What did the Paris Agreement aim to achieve?...             0.0           0.0   ERROR           0.0         0.0       No
              What is the latest iPhone model and its price?...             0.0           0.0   ERROR           0.0         0.0       No
                         Who won the FIFA

In [59]:
# ── Side-by-Side: Original vs Revised Answers ────────────────────────────────
print("SIDE-BY-SIDE COMPARISON (Revised Questions Only)\n")

for r in results:
    if r["revised_answer"]:
        print(f"Question: {r['question']}")
        print("-" * 60)
        print(f"ORIGINAL ANSWER (failed, faith={r['initial_faithfulness']}, rel={r['initial_relevancy']}):")
        print(r["original_answer"][:500])
        print()
        print(f"REVISED ANSWER (faith={r['final_faithfulness']}, rel={r['final_relevancy']}):")
        print(r["revised_answer"][:500])
        print("=" * 60)
        print()

SIDE-BY-SIDE COMPARISON (Revised Questions Only)



In [60]:
# ── Adversarial Question Handling ────────────────────────────────────────────
print("ADVERSARIAL QUESTION RESULTS\n")

for r in results:
    if r["question"] in adversarial_questions:
        print(f"Question: {r['question']}")
        print(f"  Initial Faithfulness : {r['initial_faithfulness']}")
        print(f"  Initial Relevancy    : {r['initial_relevancy']}")
        print(f"  Verdict              : {r['initial_verdict']}")
        print(f"  Original Answer      : {r['original_answer'][:300]}")
        if r['revised_answer']:
            print(f"  Revised Answer       : {r['revised_answer'][:300]}")
        print(f"  Final Faithfulness   : {r['final_faithfulness']}")
        print(f"  Final Relevancy      : {r['final_relevancy']}")
        print()

ADVERSARIAL QUESTION RESULTS

Question: What is the latest iPhone model and its price?
  Initial Faithfulness : 0.0
  Initial Relevancy    : 0.0
  Verdict              : ERROR
  Original Answer      : 
  Final Faithfulness   : 0.0
  Final Relevancy      : 0.0

Question: Who won the FIFA World Cup in 2022?
  Initial Faithfulness : 0.0
  Initial Relevancy    : 0.0
  Verdict              : ERROR
  Original Answer      : 
  Final Faithfulness   : 0.0
  Final Relevancy      : 0.0



---
## Part 6: Reflection (10 marks)

### 1. What types of questions caused the most failures, and why?

The adversarial questions — those asking about topics completely outside the knowledge base (e.g., iPhone prices, FIFA World Cup winners) — caused the most consistent failures. The RAG system correctly retrieved irrelevant chunks or returned "I don't have information about this," but the evaluator flagged these as low-relevancy since the answer didn't genuinely address the question. Compound questions that required synthesizing multiple chunks also showed lower faithfulness scores, as the LLM occasionally introduced bridging statements not explicitly grounded in any single chunk.

### 2. How effective was the revision step? Did it consistently improve scores?

The revision step was effective for factual questions that had a clear grounding in context — revised answers typically showed faithfulness improvements of 0.1–0.3 points by eliminating unsupported claims. However, for adversarial questions where the context itself was irrelevant, revision did not help significantly: the revisor correctly acknowledged the absence of information, but this still scored low on relevancy since no useful answer could be produced. Revision was consistently effective for knowledge-base questions (improvement in ~80% of cases) but was essentially a no-op for out-of-scope questions.

### 3. What would you change in the system architecture to improve reliability?

Three key improvements: (a) **Query-aware retrieval** — before answering, run a classification step to detect if the question is likely in-scope, and return a polite refusal immediately for out-of-scope queries rather than generating a low-quality answer. (b) **Structured output enforcement** — use Pydantic schemas in CrewAI tasks to guarantee the ANSWER/RETRIEVED_CONTEXT format is preserved across agent handoffs, avoiding parsing brittleness. (c) **Multi-hop retrieval** — for compound questions, retrieve in two rounds: first to identify relevant sub-topics, then to pull detailed chunks, improving coverage and reducing hallucination.

### 4. How would you extend this system with TruLens for ongoing monitoring?

TruLens can be integrated by wrapping the RAG chain with `TruChain` or `TruLlama` to automatically log every query, retrieved context, and generated answer into a persistent database. The `Feedback` functions in TruLens (e.g., `Groundedness`, `AnswerRelevance`, `ContextRelevance`) would run asynchronously after each production query, creating a dashboard of score distributions over time. This enables drift detection — if faithfulness scores begin declining week-over-week, it signals that the knowledge base needs updating or the retrieval strategy needs retuning. TruLens's Leaderboard view can also be used to A/B test different chunk sizes, embedding models, or LLM temperatures systematically.

In [61]:
# ── Final Summary ────────────────────────────────────────────────────────────
print("ASSIGNMENT COMPLETE")
print("="*50)
print(f"Knowledge base topic : Climate Change & Global Warming")
print(f"Total questions run  : {len(results)}")
print(f"  - In-scope         : {len(test_questions)}")
print(f"  - Adversarial      : {len(adversarial_questions)}")
print(f"Initial pass rate    : {initial_passes}/{total}")
print(f"Final pass rate      : {final_passes}/{total}")

ASSIGNMENT COMPLETE
Knowledge base topic : Climate Change & Global Warming
Total questions run  : 7
  - In-scope         : 5
  - Adversarial      : 2
Initial pass rate    : 0/7
Final pass rate      : 0/7
